# Feature Stores & Data Infrastructure

**Chapter 8: AI Infrastructure**

This notebook demonstrates end-to-end feature store setup using **Feast** (open-source, local) for a recommendation system.

**Running Example:**
- System: Document type recommender for InferenceBase API
- Features: User context (last 10 uploads, avg confidence) + document metadata (page count, language)
- Stores: SQLite offline store (training), Redis online store (serving)

**Tools:**
- **Feast**: Feature store framework
- **Redis**: Online store (low-latency serving)
- **Parquet**: Offline store (historical training data)

**Objectives:**
1. Define feature views (user + document features)
2. Materialize features (offline → online store)
3. Fetch online features (serving, <10ms)
4. Fetch historical features (training, point-in-time correct)
5. Measure latency (Redis vs direct DB)
6. Version features (reproduce training data)

---

## Cell 1: Setup Feast + Redis

Install Feast with Redis support and initialize feature repository.

**What we're setting up:**
- Feast CLI and Python SDK
- Redis (online store for <10ms feature lookups)
- Local Parquet offline store (training data)

**Architecture:**
```
Data Sources (Parquet)
    ↓
Feature Definitions (Python)
    ↓
Offline Store (Parquet) ← Training
Online Store (Redis)    ← Serving
```

In [ ]:
# TODO: Implement this cell


## Cell 2: Generate Synthetic User Activity Data

Create training data simulating InferenceBase user behavior:
- User uploads (document type, confidence score, page count)
- Historical events with timestamps (for point-in-time joins)

**Schema:**
- `user_id`: int (1-1000)
- `event_timestamp`: datetime
- `doc_type`: str (invoice, contract, receipt, tax_form)
- `confidence_score`: float (0.6-1.0)
- `page_count`: int (1-50)
- `has_tables`: bool

In [ ]:
# TODO: Implement this cell


## Cell 3: Define Feature Views (User + Document Features)

Create feature definitions that Feast will use to generate both training and serving features.

**Feature views:**
1. **user_activity_features** — User behavior over last 30 days
   - `avg_confidence_score`: Mean confidence of user's documents
   - `total_pages_processed`: Sum of all pages processed
   - `upload_count`: Number of documents uploaded
   - `most_common_doc_type`: User's preferred document type

2. **user_doc_type_affinity** — Per-user preference for each doc type
   - `invoice_ratio`: % of user's uploads that are invoices
   - `contract_ratio`: % contracts
   - `receipt_ratio`: % receipts
   - `tax_form_ratio`: % tax forms

In [ ]:
# TODO: Implement this cell


## Cell 4: Configure Feature Store (Redis + Parquet)

Update `feature_store.yaml` to use:
- **Online store**: Redis (localhost:6379)
- **Offline store**: Local Parquet files
- **Registry**: SQLite (feature metadata)

Then apply feature definitions to the registry.

In [ ]:
# TODO: Implement this cell


## Cell 5: Compute Features + Materialize to Online Store

**Materialization** = compute features from raw data → write to online store

**Process:**
1. Read `user_uploads.parquet`
2. Aggregate per user:
   - `AVG(confidence_score)`
   - `SUM(page_count)`
   - `COUNT(*)`
   - `doc_type` ratios
3. Write to Redis:
   - Key: `user:12345:avg_confidence_score`
   - Value: `0.87`

**Note:** In production, this runs on a schedule (hourly/daily via Airflow).

In [ ]:
# TODO: Implement this cell


## Cell 6: Fetch Online Features (Real-Time Serving)

Simulate production inference: fetch features for a single user in <10ms.

**Use case:** User 42 uploads a new document → API needs features to predict document type.

**Latency target:** <10ms (p95)

In [ ]:
# TODO: Implement this cell


## Cell 7: Fetch Historical Features (Training Data)

**Point-in-time correct joins** — fetch features as they existed at specific timestamps.

**Why this matters:**
- Prevents data leakage (model can't see future data during training)
- Reproduces exact training environment

**Example:** For a training sample at timestamp T, fetch features computed using only data BEFORE T.

In [ ]:
# TODO: Implement this cell


## Cell 8: Measure Latency — Feature Store vs Direct DB

Compare latency of:
1. **Feature store (Redis)**: Precomputed features, <10ms lookup
2. **Direct DB (simulated)**: On-the-fly aggregation, ~200-400ms

**Business impact:**
- Before: 380ms feature queries → 2.8s p95 total latency (violated SLA)
- After: 8ms feature queries → 1.4s p95 total latency (30% better than target)

In [ ]:
# TODO: Implement this cell


## Cell 9: Point-in-Time Join Validation (Prevent Data Leakage)

**Critical test:** Verify that historical features don't use future data.

**Test:**
1. Fetch features for timestamp T
2. Verify features only use data with `event_timestamp < T`
3. Confirm no "future peeking" that would inflate training accuracy

**Why this matters:** Data leakage is a silent killer — model looks great in training, fails in production.

In [ ]:
# TODO: Implement this cell


## Cell 10: Feature Versioning + Reproducibility

**Problem:** Feature definitions change over time. How do you reproduce training data from 3 months ago?

**Solution:** Git-based feature versioning.

**Workflow:**
1. Train model → tag Git commit with feature definitions
2. Log commit SHA in MLflow (from Ch.9)
3. Retrain later → checkout old Git tag → reproduce exact features

**Demo:** Show how feature definitions are version-controlled.

In [ ]:
# TODO: Implement this cell
